# Olist Logistics Agent

Foco exclusivo na análise de prazos de entrega, atrasos e performance logística para entender gargalos da cadeia de distribuição.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes de logística

In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('orders', orders.shape)
print('order_items', order_items.shape)
print('sellers', sellers.shape)

## Preparar métricas de entrega

Converter datas e calcular tempos de entrega para análise logística.

In [ ]:
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

orders['time_to_carrier'] = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.days
orders['time_to_customer'] = (orders['order_delivered_customer_date'] - orders['order_approved_at']).dt.days
orders['carrier_to_customer'] = (orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dt.days

orders[['time_to_carrier', 'time_to_customer', 'carrier_to_customer']].describe()

### Tempo por etapa logística do seller para o cliente

Neste bloco, calculamos os principais tempos da cadeia logística:
- tempo do seller até o repasse ao parceiro logístico (`order_delivered_carrier_date - order_approved_at`);
- tempo do parceiro logístico até a entrega ao cliente (`order_delivered_customer_date - order_delivered_carrier_date`);
- tempo total desde a aprovação do pedido até a entrega ao cliente;
- diferença entre `order_delivered_carrier_date` e `shipping_limit_date` para medir o cumprimento do prazo do seller;
- diferença entre `order_delivered_customer_date` e `order_estimated_delivery_date` para avaliar a entrega contra a previsão ao cliente.

A seguir, vamos comparar graficamente os valores médios de `seller_handling_days` e `logistics_days` para os sellers mais ativos.

A análise considera apenas pedidos com todas as datas necessárias preenchidas, garantindo que os tempos sejam válidos.

In [ ]:
order_sellers = pd.merge(order_items[['order_id', 'seller_id', 'shipping_limit_date']], sellers[['seller_id', 'seller_label']], on='seller_id', how='left')
order_sellers = order_sellers.drop_duplicates(subset=['order_id', 'seller_id'])
order_with_sellers = pd.merge(orders, order_sellers, on='order_id', how='left')

seller_stage_times = (
    order_with_sellers.dropna(subset=['order_delivered_carrier_date', 'order_delivered_customer_date', 'shipping_limit_date', 'order_estimated_delivery_date'])
    .assign(
        seller_handling_days=lambda df: (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.days,
        logistics_days=lambda df: (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days,
        total_days=lambda df: (df['order_delivered_customer_date'] - df['order_approved_at']).dt.days,
        actual_total_days=lambda df: df['seller_handling_days'] + df['logistics_days'],
        estimated_total_days=lambda df: (df['order_estimated_delivery_date'] - df['order_approved_at']).dt.days,
        seller_deadline_gap=lambda df: (df['order_delivered_carrier_date'] - df['shipping_limit_date']).dt.days,
        delivery_delay_vs_estimate=lambda df: (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days,
        seller_pct=lambda df: np.where(df['actual_total_days'] > 0, df['seller_handling_days'] / df['actual_total_days'], 0),
        logistics_pct=lambda df: np.where(df['actual_total_days'] > 0, df['logistics_days'] / df['actual_total_days'], 0)
    )
    .groupby('seller_label')
    .agg(
        avg_seller_handling_days=('seller_handling_days', 'mean'),
        avg_logistics_days=('logistics_days', 'mean'),
        avg_total_days=('total_days', 'mean'),
        avg_actual_total_days=('actual_total_days', 'mean'),
        avg_estimated_total_days=('estimated_total_days', 'mean'),
        avg_seller_deadline_gap=('seller_deadline_gap', 'mean'),
        avg_delivery_delay_vs_estimate=('delivery_delay_vs_estimate', 'mean'),
        avg_seller_pct=('seller_pct', 'mean'),
        avg_logistics_pct=('logistics_pct', 'mean'),
        order_count=('order_id', 'nunique')
    )
    .reset_index()
)

# Mostrar as primeiras linhas da tabela de métricas
seller_stage_times.head()

# Gráfico comparativo das principais etapas para os top 20 sellers
ranked_sellers = seller_stage_times.sort_values('order_count', ascending=False).head(20)
plot_data = ranked_sellers.melt(
    id_vars='seller_label',
    value_vars=['avg_seller_handling_days', 'avg_logistics_days'],
    var_name='stage',
    value_name='average_days'
)

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=plot_data, x='seller_label', y='average_days', hue='stage', palette='muted')
ax.set_title('Comparação média por seller: seller handling days vs logistics days')
ax.set_xlabel('Seller')
ax.set_ylabel('Dias médios')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Métrica')
plt.tight_layout()
plt.show()

# Gráfico empilhado de participação percentual do seller e da logística no tempo total real
plot_data_pct = ranked_sellers[['seller_label', 'avg_seller_pct', 'avg_logistics_pct']].copy()
plot_data_pct = plot_data_pct.sort_values('avg_logistics_pct', ascending=False)

plt.figure(figsize=(14, 7))
plt.bar(plot_data_pct['seller_label'], plot_data_pct['avg_seller_pct'], label='Seller share', color='#4c72b0')
plt.bar(
    plot_data_pct['seller_label'],
    plot_data_pct['avg_logistics_pct'],
    bottom=plot_data_pct['avg_seller_pct'],
    label='Logistics share',
    color='#d62728'
)
plt.title('Participação percentual do seller e da logística no tempo total real de entrega')
plt.xlabel('Seller')
plt.ylabel('Porcentagem do tempo total')
plt.ylim(0, 1)
plt.yticks([0, 0.25, 0.5, 0.75, 1.0], ['0%', '25%', '50%', '75%', '100%'])
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# Gráfico comparando tempo real médio vs estimado médio por seller
plot_data_time = ranked_sellers.sort_values('avg_estimated_total_days', ascending=False)
width = 0.35
x = np.arange(len(plot_data_time))

plt.figure(figsize=(14, 7))
plt.bar(x - width/2, plot_data_time['avg_actual_total_days'], width, label='Tempo real médio (dias)', color='#1f77b4')
plt.bar(x + width/2, plot_data_time['avg_estimated_total_days'], width, label='Tempo estimado médio (dias)', color='#ff7f0e')
plt.xticks(x, plot_data_time['seller_label'], rotation=45)
plt.title('Tempo real médio vs estimado médio por seller')
plt.xlabel('Seller')
plt.ylabel('Dias')
plt.legend()
plt.tight_layout()
plt.show()

# Gráfico comparando tempo real médio vs estimado médio por seller
plot_data_time = ranked_sellers.sort_values('avg_estimated_total_days', ascending=False)
width = 0.35
x = np.arange(len(plot_data_time))

plt.figure(figsize=(14, 7))
plt.bar(x - width/2, plot_data_time['avg_actual_total_days'], width, label='Tempo real médio (dias)', color='#1f77b4')
plt.bar(x + width/2, plot_data_time['avg_estimated_total_days'], width, label='Tempo estimado médio (dias)', color='#ff7f0e')
plt.xticks(x, plot_data_time['seller_label'], rotation=45)
plt.title('Tempo real médio vs estimado médio por seller')
plt.xlabel('Seller')
plt.ylabel('Dias')
plt.legend()
plt.tight_layout()
plt.show()

### Rotas críticas por estado

Agora que confirmamos que o maior tempo médio está na etapa do parceiro logístico, vamos olhar para as rotas entre sellers e clientes. Essa análise identifica quais combinações de `seller_state → customer_state` têm maior tempo médio logístico e quais possuem maior volume de pedidos.

O objetivo é encontrar rotas que combinem alto tempo médio com alto volume de pedidos, pois são esses fluxos que apontam para gargalos operacionais mais relevantes.

In [ ]:
customers = pd.read_csv(base_path + 'olist_customers_dataset.csv')[['customer_id', 'customer_state']]
order_sellers = pd.merge(
    order_items[['order_id', 'seller_id', 'shipping_limit_date']],
    sellers[['seller_id', 'seller_state']],
    on='seller_id',
    how='left'
)
order_sellers = order_sellers.drop_duplicates(subset=['order_id', 'seller_id'])
order_with_sellers = pd.merge(orders, order_sellers, on='order_id', how='left')
order_with_customer = pd.merge(order_with_sellers, customers, on='customer_id', how='left')

route_metrics = (
    order_with_customer.dropna(subset=['order_delivered_carrier_date', 'order_delivered_customer_date', 'seller_state', 'customer_state'])
    .assign(logistics_days=lambda df: (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days)
    .groupby(['seller_state', 'customer_state'])
    .agg(
        avg_logistics_days=('logistics_days', 'mean'),
        route_order_count=('order_id', 'nunique')
    )
    .reset_index()
)

route_metrics = route_metrics.assign(
    delay_volume_score=lambda df: df['avg_logistics_days'] * df['route_order_count']
)

# Top 20 rotas por score combinado de tempo médio e volume
top_routes = route_metrics.sort_values('delay_volume_score', ascending=False).head(20)

top_routes['route_label'] = top_routes['seller_state'] + ' → ' + top_routes['customer_state']

top_routes[['route_label', 'avg_logistics_days', 'route_order_count', 'delay_volume_score']].head()

plt.figure(figsize=(14, 9))
ax2 = sns.scatterplot(
    data=top_routes,
    x='avg_logistics_days',
    y='route_label',
    size='route_order_count',
    hue='route_order_count',
    palette='viridis',
    sizes=(100, 800),
    legend='brief'
)
ax2.set_title('Top 20 rotas críticas: tempo médio logístico x volume de pedidos')
ax2.set_xlabel('Dias médios de logística')
ax2.set_ylabel('Rota (seller_state → customer_state)')
plt.legend(title='Volume de pedidos', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 9))
ax3 = sns.barplot(
    data=top_routes.sort_values('delay_volume_score', ascending=False),
    x='delay_volume_score',
    y='route_label',
    palette='rocket'
)
ax3.set_title('Top 20 rotas críticas por score de tempo médio x volume')
ax3.set_xlabel('Score combinado (tempo médio x volume)')
ax3.set_ylabel('Rota (seller_state → customer_state)')
plt.tight_layout()
plt.show()

Neste gráfico, cada rota representa um par de estados de origem e destino. As rotas no topo da lista são potenciais pontos críticos, pois apresentam maior tempo médio logístico por volume de pedidos. Essas rotas podem indicar desafios operacionais de transporte regional ou pontos de transferência com atraso elevado.